# 05 — Model Selection

Membandingkan keempat kandidat (**Prophet, LSTM, GRU, TFT**) dengan membaca
tabel `{model}_val_metrics_daily` (dan `{model}_metrics_daily` utk test-set) yang ditulis
masing-masing notebook (`01_Prophet` .. `04_TFT`) ke BigQuery. Model terbaik
dipilih berdasarkan metrik agregat yang bisa dikonfigurasi — default **mean
NSE** di seluruh stasiun dan horizon, karena NSE adalah metrik standar
hidrologi yang menormalkan variansi tiap stasiun sehingga adil untuk
dibandingkan lintas lokasi.

Jalankan notebook ini setelah minimal satu (idealnya keempat) notebook model
selesai training + upload.

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

import json
import pandas as pd

PROJECT_ID = "thesis-506605"
DATASET    = "tables"
LOCATION   = "US"

MODEL_NAMES = ["prophet", "lstm", "gru", "tft"]

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)


def load_metrics_table(model_name, suffix):
    table_id = f"{PROJECT_ID}.{DATASET}.{model_name}_{suffix}_daily"
    try:
        return client.query(f"SELECT * FROM `{table_id}`").to_dataframe()
    except NotFound:
        print(f"  [skip] {table_id} belum ada \u2014 jalankan notebook {model_name} dulu.")
        return None

## Model Selection — Aggregate Validation Metrics

Semua hasil validasi dikumpulkan ke satu DataFrame panjang
(`model × station × horizon × metric`).

In [ ]:
val_frames = []
for model_name in MODEL_NAMES:
    df = load_metrics_table(model_name, "val_metrics")
    if df is not None:
        val_frames.append(df)

if not val_frames:
    raise RuntimeError("Belum ada tabel val_metrics sama sekali; jalankan notebook 01-04 dulu.")

val_long = pd.concat(val_frames, ignore_index=True)
print(f"Collected {len(val_long)} metric rows across {val_long['model'].nunique()} models: "
      f"{sorted(val_long['model'].unique().tolist())}")
val_long.head()

In [ ]:
# --- Konfigurasi pemilihan model -------------------------------------------
SELECTION_METRIC = "NSE"     # opsi: NSE | RMSE | MAE | sMAPE
HIGHER_IS_BETTER = {"NSE": True, "RMSE": False, "MAE": False, "sMAPE": False}

agg = (val_long[val_long["metric"] == SELECTION_METRIC]
       .groupby("model")["value"]
       .mean()
       .sort_values(ascending=not HIGHER_IS_BETTER[SELECTION_METRIC]))

print(f"Mean {SELECTION_METRIC} per model (validation):")
for m, v in agg.items():
    print(f"  {m:<8} {v:>8.4f}")

BEST_MODEL = agg.index[0]
BEST_VALUE = float(agg.iloc[0])
print(f"\n\u2714 Best model by mean {SELECTION_METRIC}: '{BEST_MODEL}' ({SELECTION_METRIC}={BEST_VALUE:.4f})")

# Ringkasan pivot (model x horizon) untuk metrik terpilih
pivot = (val_long[val_long["metric"] == SELECTION_METRIC]
         .pivot_table(index="model", columns="horizon", values="value", aggfunc="mean"))
pivot

## Best Metrics — Detil Model Terbaik

Semua metrik (`RMSE`, `MAE`, `sMAPE`, `NSE`) per stasiun × horizon untuk
`BEST_MODEL`, di validation set dan (bila sudah tersedia) test set.

In [ ]:
best_val_pivot = (val_long[val_long["model"] == BEST_MODEL]
                   .pivot_table(index=["station", "horizon"], columns="metric", values="value"))
print(f"Metrik validasi \u2014 '{BEST_MODEL}':")
best_val_pivot

In [ ]:
test_frames = []
for model_name in MODEL_NAMES:
    df = load_metrics_table(model_name, "metrics")
    if df is not None:
        test_frames.append(df)

if test_frames:
    test_long = pd.concat(test_frames, ignore_index=True)

    agg_test = (test_long[test_long["metric"] == SELECTION_METRIC]
                .groupby("model")["value"]
                .mean()
                .sort_values(ascending=not HIGHER_IS_BETTER[SELECTION_METRIC]))
    print(f"Mean {SELECTION_METRIC} per model (test set):")
    for m, v in agg_test.items():
        print(f"  {m:<8} {v:>8.4f}")

    best_test_pivot = (test_long[test_long["model"] == BEST_MODEL]
                        .pivot_table(index=["station", "horizon"], columns="metric", values="value"))
    print(f"\nMetrik test-set \u2014 '{BEST_MODEL}':")
    print(best_test_pivot)
else:
    test_long = pd.DataFrame()
    print("Belum ada tabel test metrics; jalankan notebook model (01-04) sampai selesai (termasuk sel test-set evaluation) dulu.")

## Persist Model Selection to BigQuery

Hasil pemilihan (`best_model`, metrik yang dipakai, nilai agregatnya, dan
daftar model yang dibandingkan) disimpan sebagai satu baris di
`model_selection_daily`, sehingga keputusan ini punya jejak permanen dan bisa
dibaca ulang tanpa menjalankan ulang notebook ini.

In [ ]:
SELECTION_TABLE = f"{PROJECT_ID}.{DATASET}.model_selection_daily"

selection_schema = [
    bigquery.SchemaField("selection_metric", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("best_model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("mean_value", "FLOAT", mode="REQUIRED"),
    bigquery.SchemaField("models_compared", "STRING", mode="REQUIRED"),   # JSON list
    bigquery.SchemaField("saved_at", "TIMESTAMP", mode="REQUIRED"),
]

selection_df = pd.DataFrame([{
    "selection_metric": SELECTION_METRIC,
    "best_model": BEST_MODEL,
    "mean_value": BEST_VALUE,
    "models_compared": json.dumps(sorted(val_long["model"].unique().tolist())),
    "saved_at": pd.Timestamp.now(tz="UTC").tz_localize(None),
}])

job_config = bigquery.LoadJobConfig(schema=selection_schema, write_disposition="WRITE_TRUNCATE")
job = client.load_table_from_dataframe(selection_df, SELECTION_TABLE, job_config=job_config)
job.result()

print(f"\n\u2714 Model selection tersimpan di {SELECTION_TABLE}: "
      f"best_model='{BEST_MODEL}' ({SELECTION_METRIC}={BEST_VALUE:.4f})")